In [1]:
import gzip
import json
import numpy as np
import math
import re
import implementation as impl
from collections import Counter

STUDENT IMPLEMENTATION
Vocabulary:       ['cat', 'dog', 'eats', 'fish', 'likes']
Counts D1:        [1 0 1 1 0]
TF D1:            [0.3333 0.     0.3333 0.3333 0.    ]
IDF:              [0.4055 1.0986 0.4055 0.     1.0986]
TF-IDF D1:        [0.1352 0.     0.1352 0.     0.    ]
Cosine sim [1,1,1] vs [1,1,0]: 0.8165

UNIT TESTS
[PASS] build_vocabulary
[PASS] compute_counts
[PASS] compute_tf
[PASS] compute_idf
[PASS] compute_tfidf
[PASS] cosine_similarity

All unit tests passed!

SO SÁNH: STUDENT vs SKLEARN (default settings)
Sklearn vocabulary: ['cat', 'dog', 'eats', 'fish', 'likes']
Sklearn TF-IDF D1:  [0.6198 0.     0.6198 0.4813 0.    ]
Student TF-IDF D1:  [0.1352 0.     0.1352 0.     0.    ]
>> Khác nhau do: sklearn mặc định dùng IDF smoothing + L2 normalization

SO SÁNH: STUDENT vs SKLEARN (tắt smoothing + normalization)
Sklearn vocabulary: ['cat', 'dog', 'eats', 'fish', 'likes']
Sklearn TF-IDF D1:  [1.4055 0.     1.4055 1.     0.    ]
Student TF-IDF D1:  [0.1352 0.     0.1352 0.     

In [2]:
path = "D:\\Study\\Study_Class\\Semester_7\\NLP\\Lab\\lab01\\c4-train.00000-of-01024-30K.json.gz"

with gzip.open(path, "rt", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

print(len(data))
print(data[0])

30000
{'text': 'Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.', 'timestamp': '2019-04-25T12:57:54Z', 'url': 'https://klyq.com/beginners-bbq-class-taking-place-in-missoula/'}


# Part D 

## 7.2

In [3]:
documents = [item["text"] for item in data]

print("number of docs: ", len(documents))
print("example the first doc: ", documents[0])

number of docs:  30000
example the first doc:  Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.
The cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.


In [4]:
vocabulary = impl.build_vocabulary(documents)
N = len(documents)
V = len(vocabulary)

print(f"N (documents) = {N}")
print(f"V (vocabulary) = {V}")

N (documents) = 30000
V (vocabulary) = 193837


In [5]:
print(f"Number of documents = {N}")
print(f"Vocabulary size     = {V}")
print(f"Matrix shape         = ({N}, {V})")

Number of documents = 30000
Vocabulary size     = 193837
Matrix shape         = (30000, 193837)


## 7.4 

In [6]:
term_to_index = {term: i for i, term in enumerate(vocabulary)}
nnz = 0
for doc in documents:
    tokens = set(re.findall(r'\b\w+\b', doc.lower()))
    nnz += sum(1 for term in tokens if term in term_to_index)

S = 1 - nnz /(N * V)
print(f"nnz(X) = {nnz}")
print(f"N x V = {N * V}")
print(f"Sparsity S = {S:.6f}")

nnz(X) = 5104560
N x V = 5815110000
Sparsity S = 0.999122


## 7.5

In [7]:
df_counts = np.zeros(V, dtype=int)

for doc in documents:
    tokens = set(re.findall(r'\b\w+\b', doc.lower()))
    for term in tokens:
        if term in term_to_index:
            df_counts[term_to_index[term]] += 1

idf = np.array([math.log(N / df) if df > 0 else 0.0 for df in df_counts])

top_df_idx = np.argsort(df_counts)[::-1][:20]
print("the 20 most common tearms by DF")
for i in top_df_idx:
    print(f"{vocabulary[i]:20s} df={df_counts[i]}")

top_idf_idx = np.argsort(idf)[::-1][:20]
print("\nThe 20 terms with the highest IDF")
for i in top_idf_idx:
    print(f"{vocabulary[i]:20s} idf={idf[i]:.4f}")


doc_index = 0   # chọn document đầu tiên, có thể đổi số này
chosen_doc = documents[doc_index]

tokens = re.findall(r'\b\w+\b', chosen_doc.lower())
counter = Counter(tokens)
total_tokens = sum(counter.values())

tfidf_doc = {}
for term, c in counter.items():
    if term in term_to_index:
        idx = term_to_index[term]
        tf_val = c / total_tokens
        tfidf_doc[idx] = tf_val * idf[idx]

top_tfidf_idx = sorted(tfidf_doc.keys(), key=lambda i: tfidf_doc[i], reverse=True)[:20]
print(f"\nThe 20 terms with the highest TF-IDF in the document. {doc_index}")
for i in top_tfidf_idx:
    print(f"{vocabulary[i]:20s} tfidf={tfidf_doc[i]:.4f}")

the 20 most common tearms by DF
the                  df=27893
and                  df=27423
to                   df=26689
of                   df=26031
a                    df=25905
in                   df=25224
for                  df=23651
is                   df=22739
with                 df=21405
on                   df=20262
that                 df=18370
this                 df=17840
are                  df=17594
it                   df=17168
s                    df=16959
as                   df=16467
at                   df=16347
from                 df=16316
be                   df=16153
you                  df=16094

The 20 terms with the highest IDF
𐌼𐌿𐌽𐌳𐍃                idf=10.3090
hidi                 idf=10.3090
hingus               idf=10.3090
hingucker            idf=10.3090
hinesc               idf=10.3090
hindware             idf=10.3090
hindutva             idf=10.3090
hindustani           idf=10.3090
hindrichs            idf=10.3090
hindraf              idf=10.3090
hin